# Maize GxE ML Pipeline (Notebook Conversion)

This notebook is a cell-structured conversion of `maize_gxe_ml_pipeline.py`.

It preserves the original pipeline logic, assumptions, helper functions, modeling flow, and output artifacts while making execution easier to run step by step.

Core assumptions retained:
- Analytical unit: `LINE × (LOC × YEAR)`
- Environment key: `env_id = LOC_YEAR`
- Line key: `line_id` derived from `LINE_UNIQUE_ID` formats like `C1.1.191`
- Primary target: `YLDBE` (from `YLD_BE` / `YLDBE` columns)
- Genomic data from cohort/population-specific imputed files (e.g., `C1.<pop>_Imputed.csv`)


In [1]:
"""
Maize line yield (YLDBE) under GxE — full ML pipeline.

## 1. Lock Analytical Unit
One row = LINE × (LOC × YEAR); keys `env_id` = LOC_YEAR, `line_id` = LINE_UNIQUE_ID
(e.g. C1.1.191). Target YLDBE (`YLD_BE` in files). Alternatives: PHT, MST.

## 2. Build Genotype Matrix
Per-pop imputed files; progeny rows match ^\d{11}$; line_id = C{cohort}.{pop}.{linenum}.

## 3. Validate Joins
Match rates, past_yld from train-year aggregates, NaN column drops, geno coverage check.

## 4. Refine Genetic Encoding
PCA (50 comps, refit inside CV on train-fold lines) + Ridge(alpha=1e-2) on raw SNPs baseline.

## 5. Environment Covariates
Monthly PRCP/TAVG-style columns, totals, summer averages, global StandardScaler.

## 6. Improve GxE Features
Domain blocks + RF permutation screen (top 5/block, cap 20 env vars) + PCA×env interactions.

## 7. Gradient Boosting
Phase A: RandomizedSearchCV + GroupKFold on GBR. Phase B: HistGradientBoosting (or optional XGB)
with year-holdout early stopping.

## 8. Unbiased Evaluation
OOF metrics global and by env_id; residual plot; per-env R² table.

## 9. Ranking Under Constraints
Top-k per env; bootstrap uncertainty; optional genetic distance diversification.

## 10. Interpretation
Permutation importance by bucket; optional SHAP summary for tree model.

## 11. Packaging
YAML config + run_summary.json under output/.
"""

from __future__ import annotations

import argparse
import json
import logging
import re
import sys
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import joblib
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
import time

import torch
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

PROGENY_RE = re.compile(r"^\d{11}$")
LINE_ID_RE = re.compile(r"^C(\d+)\.(\d+)\.(\d+)$", re.IGNORECASE)
POP_FILE_RE_C1 = re.compile(r"C1\.(\d+)_Imputed\.csv$", re.IGNORECASE)
POP_FILE_RE_C2 = re.compile(r"C2\.(\d+)_Imputed\.csv$", re.IGNORECASE)


## Utilities and Core Helpers

Logging, config loading, column inference, and metrics utilities from the script.

In [2]:
def setup_logging(output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    log_path = output_dir / "pipeline.log"
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
    root = logging.getLogger()
    root.setLevel(logging.INFO)
    root.handlers.clear()
    fh = logging.FileHandler(log_path, encoding="utf-8")
    fh.setFormatter(fmt)
    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    root.addHandler(fh)
    root.addHandler(sh)


def load_config(path: Path) -> Dict[str, Any]:
    with open(path, encoding="utf-8") as f:
        return yaml.safe_load(f)


def log_step(name: str) -> None:
    logging.info("=== %s ===", name)


def infer_col(cols: Sequence[str], candidates: Sequence[str]) -> str:
    up = {str(c).upper(): c for c in cols}
    for cand in candidates:
        if cand.upper() in up:
            return up[cand.upper()]
    raise ValueError(f"None of {candidates} found in columns {list(cols)[:20]}...")


def metrics_dict(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    if m.sum() < 2:
        return {"rmse": float("nan"), "mae": float("nan"), "r2": float("nan")}
    yt, yp = y_true[m], y_pred[m]
    return {
        "rmse": float(np.sqrt(mean_squared_error(yt, yp))),
        "mae": float(mean_absolute_error(yt, yp)),
        "r2": float(r2_score(yt, yp)),
    }


## Data Loading and Key Joins

Step 1 locks the analytical grain and joins phenotype with environment on `YEAR` and `LOC`.

Step 2 builds the genotype matrix from population-imputed files.

Step 3 validates join quality and prepares the integrated modeling frame.

In [3]:
# --- Step 1 ---
def step1_lock_analytical_unit(
    pheno_path: Path,
    env_path: Path,
    target_col: str,
    output_dir: Path,
) -> pd.DataFrame:
    log_step("Step 1: Lock analytical unit")
    pheno = pd.read_csv(pheno_path, low_memory=False)
    # Harmonize year column
    year_col = "YEAR_x" if "YEAR_x" in pheno.columns else infer_col(pheno.columns, ["YEAR"])
    loc_col = infer_col(pheno.columns, ["LOC"])
    line_col = infer_col(pheno.columns, ["LINE_UNIQUE_ID", "LINEUNIQUEID"])
    pheno = pheno.rename(columns={year_col: "YEAR", loc_col: "LOC", line_col: "LINE_UNIQUE_ID_RAW"})
    pheno["LINE_UNIQUE_ID"] = pheno["LINE_UNIQUE_ID_RAW"].astype(str).str.strip()
    pheno["YEAR"] = pd.to_numeric(pheno["YEAR"], errors="coerce")
    pheno["LOC"] = pheno["LOC"].astype(str).str.strip()
    pheno["env_id"] = pheno["LOC"] + "_" + pheno["YEAR"].astype("Int64").astype(str)

    extr = pheno["LINE_UNIQUE_ID"].str.extract(LINE_ID_RE)
    extr.columns = ["cohort_num", "pop_num", "line_num"]
    for c in extr.columns:
        pheno[c] = pd.to_numeric(extr[c], errors="coerce")
    bad = pheno["cohort_num"].isna()
    if bad.any():
        logging.warning("Dropped %s phenotype rows with non-parseable line_id", int(bad.sum()))
        pheno = pheno.loc[~bad].copy()

    # Canonical line_id (e.g. C1.114.00000000001 -> C1.114.1) for genotype join
    pheno["line_id"] = (
        "C"
        + pheno["cohort_num"].astype(int).astype(str)
        + "."
        + pheno["pop_num"].astype(int).astype(str)
        + "."
        + pheno["line_num"].astype(int).astype(str)
    )

    tgt = infer_col(pheno.columns, [target_col, "YLDBE", "YLD_BE"])
    pheno["YLDBE"] = pd.to_numeric(pheno[tgt], errors="coerce")

    key = ["line_id", "env_id"]
    dup_mask = pheno.duplicated(key, keep=False)
    if dup_mask.any():
        n = int(dup_mask.sum())
        logging.warning("Duplicate grain rows: %s - dropping duplicates (keep first)", n)
        pheno = pheno.drop_duplicates(subset=key, keep="first")

    env = pd.read_csv(env_path, low_memory=False)
    ey = infer_col(env.columns, ["YEAR"])
    el = infer_col(env.columns, ["LOC"])
    env = env.rename(columns={ey: "YEAR", el: "LOC"})
    env["YEAR"] = pd.to_numeric(env["YEAR"], errors="coerce")
    env["LOC"] = env["LOC"].astype(str).str.strip()

    merged = pheno.merge(env, on=["YEAR", "LOC"], how="left", suffixes=("", "_ENV"))
    out_path = output_dir / "pheno_env.csv"
    merged.to_csv(out_path, index=False)
    logging.info("Saved %s rows to %s", len(merged), out_path)
    return merged


In [4]:
# --- Step 2 ---
def step2_build_genotype_matrix(
    imputed_dir: Path,
    cohort: str,
    output_dir: Path,
    max_snps: Optional[int],
    max_pop_files: Optional[int],
) -> pd.DataFrame:
    log_step("Step 2: Build genotype matrix")
    cohort = cohort.upper()
    pat = POP_FILE_RE_C1 if cohort == "C1" else POP_FILE_RE_C2
    files = [f for f in imputed_dir.glob("*_Imputed.csv") if pat.search(f.name)]
    files.sort(key=lambda p: int(pat.search(p.name).group(1)))
    if max_pop_files is not None:
        files = files[: int(max_pop_files)]
    if not files:
        raise FileNotFoundError(f"No imputed CSVs matching {pat.pattern} under {imputed_dir}")

    snp_cols_ref: Optional[List[str]] = None
    chunks: List[pd.DataFrame] = []

    for i, fp in enumerate(files):
        m = pat.search(fp.name)
        if not m:
            continue
        pop = int(m.group(1))
        df = pd.read_csv(fp, index_col=0, low_memory=False)
        df.index = df.index.astype(str)
        prog_mask = df.index.str.match(PROGENY_RE)
        df = df.loc[prog_mask].copy()
        if df.empty:
            continue
        if snp_cols_ref is None:
            snp_cols_ref = [c for c in df.columns]
            if max_snps is not None and len(snp_cols_ref) > max_snps:
                rng = np.random.default_rng(42)
                snp_cols_ref = list(rng.choice(snp_cols_ref, size=max_snps, replace=False))
        else:
            df = df[[c for c in snp_cols_ref if c in df.columns]]
            missing = set(snp_cols_ref) - set(df.columns)
            if missing:
                for c in missing:
                    df[c] = np.nan

        df = df[snp_cols_ref]
        linenum = df.index.to_series().map(lambda s: int(s.lstrip("0") or "0"))
        cohort_digit = "1" if cohort == "C1" else "2"
        df.insert(0, "line_id", [f"C{cohort_digit}.{pop}.{ln}" for ln in linenum])
        df = df.drop_duplicates(subset=["line_id"], keep="first")
        chunks.append(df)
        if (i + 1) % 50 == 0:
            logging.info("Loaded %s/%s population files", i + 1, len(files))

    geno = pd.concat(chunks, axis=0, ignore_index=False)
    geno = geno.reset_index(drop=True)
    snp_names = [c for c in geno.columns if c != "line_id"]
    X = geno[snp_names].apply(pd.to_numeric, errors="coerce").astype(np.float32)
    X = X.fillna(0.0)
    geno[snp_names] = X
    out = output_dir / f"genotypes_{cohort.lower()}.csv"
    geno.to_csv(out, index=False)
    logging.info("Genotype matrix: %s lines, %s SNPs -> %s", len(geno), len(snp_names), out)
    return geno


In [5]:
# --- Step 3 ---
def step3_validate_joins(
    pheno_env: pd.DataFrame,
    geno: pd.DataFrame,
    train_min: int,
    train_max: int,
    output_dir: Path,
    min_geno_rate: float,
    min_pheno_env_rate: float,
    nan_drop_frac: float,
    subset_populations: bool,
    save_full_dataset_csv: bool = True,
) -> Tuple[pd.DataFrame, List[str], Dict[str, Any]]:
    log_step("Step 3: Validate joins")
    snp_cols = [c for c in geno.columns if c != "line_id"]
    pheno_lines = set(pheno_env["line_id"].unique())
    geno_lines = set(geno["line_id"].unique())
    matched = pheno_lines & geno_lines

    pops_with_geno: set[int] = set()
    for lid in geno_lines:
        m = LINE_ID_RE.match(str(lid))
        if m:
            pops_with_geno.add(int(m.group(2)))

    if subset_populations and pops_with_geno:
        extr = pheno_env["line_id"].astype(str).str.extract(LINE_ID_RE)
        pop_ser = pd.to_numeric(extr[1], errors="coerce")
        in_loaded_pop = pop_ser.isin(pops_with_geno).values
        pheno_lines_scoped = set(pheno_env.loc[in_loaded_pop, "line_id"].unique())
        denom = max(len(pheno_lines_scoped), 1)
        rate = len(matched) / denom
        logging.info(
            "Genotype subset mode: evaluating coverage among %s pheno lines in loaded populations only",
            len(pheno_lines_scoped),
        )
    else:
        denom = max(len(pheno_lines), 1)
        rate = len(matched) / denom

    logging.info("Unique phenotype lines: %s", len(pheno_lines))
    logging.info("Unique genotype lines: %s", len(geno_lines))
    logging.info("Overlap lines: %s (%.2f%% of evaluation set)", len(matched), 100 * rate)
    if rate < min_geno_rate:
        raise RuntimeError(
            f"Genotype coverage {rate:.2%} < required {min_geno_rate:.2%}. "
            "Check cohort/paths and line_id parsing."
        )

    env_keys = pheno_env[["YEAR", "LOC"]].drop_duplicates()
    env_present = pheno_env.dropna(subset=[c for c in pheno_env.columns if c not in {"YLDBE"}][:1])
    # Phenotype→env: rows where key env columns exist (not all-NaN from failed merge)
    meta_env_cols = [c for c in pheno_env.columns if any(x in c.upper() for x in ("PRCP", "TAVG", "CLAY", "SILT", "PHH2O"))]
    if meta_env_cols:
        ok = pheno_env[meta_env_cols].notna().any(axis=1)
        pe_rate = float(ok.mean())
    else:
        ok = pd.Series(True, index=pheno_env.index)
        pe_rate = 1.0
    logging.info("Phenotype rows with any env feature: %.2f%%", 100 * pe_rate)
    if pe_rate < min_pheno_env_rate:
        logging.warning(
            "Phenotype-env match %.2f%% below target %.2f%% - continuing with warning",
            100 * pe_rate,
            100 * min_pheno_env_rate,
        )

    unmatched_pheno = sorted(pheno_lines - geno_lines)[:10]
    unmatched_geno = sorted(geno_lines - pheno_lines)[:10]
    logging.info("Top unmatched pheno line_ids: %s", unmatched_pheno)
    logging.info("Top unmatched geno line_ids (sample): %s", unmatched_geno)

    train_mask = pheno_env["YEAR"].between(train_min, train_max) & pheno_env["YLDBE"].notna()
    past = (
        pheno_env.loc[train_mask]
        .groupby("line_id", as_index=False)["YLDBE"]
        .agg(past_yld_mean="mean", past_yld_median="median")
    )

    # Narrow merge only (wide SNP matrix stays in genotypes_*.csv — avoids TB-scale CSV).
    full = pheno_env.merge(past, on="line_id", how="left")
    full["has_genotype"] = full["line_id"].isin(geno_lines)

    nan_frac = full.isna().mean()
    drop_cols = nan_frac[nan_frac > nan_drop_frac].index.tolist()
    drop_cols = [c for c in drop_cols if c not in {"line_id", "env_id", "YEAR", "LOC", "YLDBE"}]
    if drop_cols:
        logging.info("Dropping %s columns with >%.0f%% NaN", len(drop_cols), nan_drop_frac * 100)
        full = full.drop(columns=drop_cols, errors="ignore")

    full_path = output_dir / "full_dataset.csv"
    if not save_full_dataset_csv:
        logging.info("Skipping full_dataset.csv write (save_full_dataset_csv=false)")
    else:
        full.to_csv(full_path, index=False)
        logging.info("Saved full_dataset.csv (%s rows)", len(full))

    report = {
        "geno_match_rate": rate,
        "pheno_env_row_rate": pe_rate,
        "unmatched_pheno_top10": unmatched_pheno,
        "unmatched_geno_top10": unmatched_geno,
        "dropped_high_nan_cols": drop_cols,
    }
    return full, snp_cols, report


## Environment Features and GxE Feature Selection

Environment scaling/aggregation, domain block definitions, and selection of environment + interaction candidates.

In [6]:
# --- Step 5 (env feats) — used before modeling ---
def step5_environment_features(df: pd.DataFrame, scaling: str) -> Tuple[pd.DataFrame, List[str], StandardScaler]:
    log_step("Step 5: Environment covariates")
    out = df.copy()
    meta = {"line_id", "env_id", "YEAR", "LOC", "YLDBE", "past_yld_mean", "past_yld_median"}
    meta |= {c for c in out.columns if c.startswith("cohort") or c.endswith("_num")}

    prcp_cols = [c for c in out.columns if "PRCP" in c.upper()]
    tavg_cols = [c for c in out.columns if "TAVG" in c.upper()]
    if prcp_cols:
        out["total_PRCP"] = out[prcp_cols].apply(pd.to_numeric, errors="coerce").sum(axis=1)
    summer_cols = [c for c in out.columns if re.search(r"(^|[^0-9])(6|7|8)[^0-9]*TAVG", c.upper()) or c.upper() in {"X06_TAVG", "X07_TAVG", "X08_TAVG"}]
    if not summer_cols:
        summer_cols = [c for c in tavg_cols if any(m in c for m in ("06", "07", "08", "6_", "7_", "8_"))]
    if summer_cols:
        out["summer_tavg"] = out[summer_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)

    env_numeric = []
    for c in out.columns:
        if c in meta or c.startswith("SNP_"):
            continue
        if out[c].dtype == object:
            continue
        if pd.api.types.is_numeric_dtype(out[c]):
            env_numeric.append(c)

    scaler = StandardScaler()
    if scaling.strip().lower() == "global" and env_numeric:
        out[env_numeric] = scaler.fit_transform(out[env_numeric].apply(pd.to_numeric, errors="coerce").fillna(0.0))
    return out, env_numeric, scaler


def domain_blocks(env_cols: List[str]) -> Dict[str, List[str]]:
    blocks = {"precip": [], "temp": [], "soil": [], "other": []}
    for c in env_cols:
        u = c.upper()
        if "PRCP" in u or "DP01" in u or "DP10" in u:
            blocks["precip"].append(c)
        elif "TAVG" in u or "HTDD" in u or "CLDD" in u or "TEMP" in u:
            blocks["temp"].append(c)
        elif any(x in u for x in ("CLAY", "SILT", "SAND", "PHH2O", "SOC", "NITROGEN", "CFVO")):
            blocks["soil"].append(c)
        else:
            blocks["other"].append(c)
    return blocks


# --- Step 6 ---
def step6_select_env_and_gxe(
    df: pd.DataFrame,
    env_numeric: List[str],
    gpc_cols: List[str],
    rng: np.random.Generator,
    max_env: int,
    n_pairs: int,
) -> Tuple[List[str], List[Tuple[str, str]]]:
    log_step("Step 6: GxE feature refinement")
    blocks = domain_blocks(env_numeric)
    cheap = RandomForestRegressor(
        n_estimators=60,
        max_depth=6,
        random_state=int(rng.integers(0, 1_000_000)),
        n_jobs=1,
    )
    y = df["YLDBE"].values
    chosen: List[str] = []

    for _, cols in blocks.items():
        cols = [c for c in cols if c in df.columns and df[c].notna().any()]
        if len(cols) <= 5:
            chosen.extend(cols)
            continue
        Xb = df[cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)
        if Xb.shape[1] == 0:
            continue
        cheap.fit(Xb, y)
        imp = permutation_importance(cheap, Xb, y, n_repeats=5, random_state=42, n_jobs=1)
        order = np.argsort(-imp.importances_mean)[:5]
        chosen.extend([cols[i] for i in order])

    chosen = list(dict.fromkeys(chosen))
    chosen = chosen[:max_env]

    pair_list: List[Tuple[str, str]] = []
    if gpc_cols and chosen:
        use_g = gpc_cols[:10]
        use_e = chosen[:10]
        Xp = df[use_g + use_e].apply(pd.to_numeric, errors="coerce").fillna(0.0)
        cheap.fit(Xp, y)
        imp = permutation_importance(cheap, Xp, y, n_repeats=4, random_state=42, n_jobs=1)
        names = list(Xp.columns)
        scored: List[Tuple[str, str, float]] = []
        for gc in use_g:
            for ec in use_e:
                if gc not in names or ec not in names:
                    continue
                i, j = names.index(gc), names.index(ec)
                scored.append((gc, ec, float(imp.importances_mean[i] + imp.importances_mean[j])))
        scored.sort(key=lambda t: -t[2])
        pair_list = [(a, b) for a, b, _ in scored[:n_pairs]]
    return chosen, pair_list


## GPU PCA (PyTorch `pca_lowrank`)

SNP→GPC uses **CUDA** when `use_gpu: true` and PyTorch sees CUDA. YAML: `max_snps`, `pca_once`, `gpu_pca_n_folds`, `cuda_device`.


In [7]:
import time

import torch

from sklearn.decomposition import PCA


def _effective_pca_k(n_samples: int, n_features: int, requested: int) -> int:
    return max(1, min(int(requested), n_samples - 1, n_features))


def pca_torch_gpu(
    X_t: torch.Tensor,
    n_components: int,
    center: bool = True,
) -> tuple:
    """GPU PCA via torch.pca_lowrank. Returns (scores_np, V_np, S_np)."""
    n, p = X_t.shape
    k = _effective_pca_k(n, p, n_components)
    if center:
        mean = X_t.mean(dim=0, keepdim=True)
        Xc = X_t - mean
    else:
        mean = torch.zeros(1, p, device=X_t.device, dtype=X_t.dtype)
        Xc = X_t
    _U, S, V = torch.pca_lowrank(Xc.float(), q=k, center=False, niter=5)
    Vk = V[:, :k]
    scores = Xc @ Vk
    return scores.cpu().numpy(), Vk.cpu().numpy(), S[:k].cpu().numpy()


def pca_torch_gpu_fit(
    X_t: torch.Tensor,
    n_components: int,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """Returns scores (N,K), V (P,K), S (K), mean (1,P) on device."""
    n, p = X_t.shape
    k = _effective_pca_k(n, p, n_components)
    mean = X_t.mean(dim=0, keepdim=True)
    Xc = X_t - mean
    _U, S, V = torch.pca_lowrank(Xc.float(), q=k, center=False, niter=5)
    Vk = V[:, :k]
    scores = Xc @ Vk
    return scores, Vk, S[:k], mean


def pca_torch_project(X_t: torch.Tensor, mean: torch.Tensor, Vk: torch.Tensor) -> torch.Tensor:
    return (X_t - mean) @ Vk


def build_X_snp_pruned_gpu(
    model_df: pd.DataFrame,
    geno_wide: pd.DataFrame,
    snp_cols: List[str],
    max_snps: int,
    device: torch.device,
) -> tuple[torch.Tensor, List[str], torch.Tensor]:
    X_np = (
        model_df[["line_id"]]
        .merge(geno_wide, on="line_id", how="left")[snp_cols]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0.0)
        .values.astype(np.float32)
    )
    X_t = torch.from_numpy(X_np).to(device, non_blocking=True)
    p = X_t.shape[1]
    k = min(max_snps, p)
    snp_var = torch.var(X_t, dim=0, unbiased=False)
    top_idx = torch.topk(snp_var, k).indices
    top_idx = torch.sort(top_idx).values
    Xp = X_t[:, top_idx]
    pruned_cols = [snp_cols[int(i)] for i in top_idx.cpu().numpy()]
    logging.info("GPU SNPs pruned: %s (from %s)", Xp.shape[1], p)
    return Xp, pruned_cols, top_idx


def fit_pca_sklearn_cpu(
    geno_wide: pd.DataFrame,
    snp_cols_sub: Sequence[str],
    line_ids: np.ndarray,
    n_components: int,
    random_state: int,
) -> Pipeline:
    sub = geno_wide.loc[geno_wide["line_id"].isin(set(line_ids)), ["line_id"] + list(snp_cols_sub)]
    sub = sub.drop_duplicates("line_id")
    X = np.nan_to_num(sub[list(snp_cols_sub)].values.astype(np.float32), nan=0.0)
    n_comp = _effective_pca_k(X.shape[0], X.shape[1], n_components)
    pipe = Pipeline(
        [
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components=n_comp, random_state=random_state)),
        ]
    )
    pipe.fit(X)
    return pipe


def map_pca_sklearn_to_rows(
    geno_wide: pd.DataFrame,
    snp_cols_sub: Sequence[str],
    line_ids: pd.Series,
    pipe: Pipeline,
) -> pd.DataFrame:
    sub = geno_wide[["line_id"] + list(snp_cols_sub)].drop_duplicates("line_id").set_index("line_id")
    X = sub.reindex(line_ids).astype(np.float32)
    X = np.nan_to_num(X.values, nan=0.0)
    comps = pipe.transform(X)
    gpc_cols = [f"GPC_{i+1}" for i in range(comps.shape[1])]
    return pd.DataFrame(comps, columns=gpc_cols, index=line_ids.index)


def _unique_train_snp_tensor(
    geno_wide: pd.DataFrame,
    snp_cols_sub: List[str],
    line_ids: np.ndarray,
    device: torch.device,
) -> torch.Tensor:
    sub = geno_wide.loc[geno_wide["line_id"].isin(set(line_ids)), ["line_id"] + snp_cols_sub]
    sub = sub.drop_duplicates("line_id")
    X = np.nan_to_num(sub[snp_cols_sub].values.astype(np.float32), nan=0.0)
    return torch.from_numpy(X).to(device, non_blocking=True)


def grouped_oof_with_pca(
    df: pd.DataFrame,
    snp_cols: List[str],
    geno_wide: pd.DataFrame,
    nongeno_feature_cols: List[str],
    y: np.ndarray,
    groups: np.ndarray,
    n_splits: int,
    pca_n: int,
    model,
    random_state: int,
    *,
    snp_cols_pruned: Optional[List[str]] = None,
    X_snp_pruned_gpu: Optional[torch.Tensor] = None,
    model_row_positions: Optional[np.ndarray] = None,
    use_gpu: bool = False,
    pca_once: bool = False,
    Vk_frozen: Optional[torch.Tensor] = None,
    mean_frozen: Optional[torch.Tensor] = None,
) -> np.ndarray:
    cols = snp_cols_pruned if snp_cols_pruned is not None else snp_cols
    gkf = GroupKFold(n_splits=n_splits)
    oof = np.full(len(y), np.nan)

    if use_gpu and X_snp_pruned_gpu is not None and model_row_positions is not None:
        assert len(model_row_positions) == len(df)
        dev = X_snp_pruned_gpu.device
        for fold, (tr, te) in enumerate(gkf.split(df, y, groups=groups)):
            t_fold = time.time()
            if pca_once and Vk_frozen is not None and mean_frozen is not None:
                Vk, mean_f = Vk_frozen, mean_frozen
            else:
                tr_lines = df.iloc[tr]["line_id"].unique()
                X_fit_t = _unique_train_snp_tensor(geno_wide, cols, tr_lines, dev)
                k_eff = _effective_pca_k(X_fit_t.shape[0], X_fit_t.shape[1], pca_n)
                _sc, Vk, _S, mean_f = pca_torch_gpu_fit(X_fit_t, k_eff)
                del X_fit_t
                if dev.type == "cuda":
                    torch.cuda.empty_cache()
            ix_tr = model_row_positions[tr]
            ix_te = model_row_positions[te]
            gpc_tr = pca_torch_project(X_snp_pruned_gpu[ix_tr], mean_f, Vk).cpu().numpy()
            gpc_te = pca_torch_project(X_snp_pruned_gpu[ix_te], mean_f, Vk).cpu().numpy()
            Xtr = np.hstack([gpc_tr, df.iloc[tr][nongeno_feature_cols].values.astype(float)])
            Xte = np.hstack([gpc_te, df.iloc[te][nongeno_feature_cols].values.astype(float)])
            model.fit(Xtr, y[tr])
            oof[te] = model.predict(Xte)
            logging.info("GPU PCA OOF fold %s/%s in %.2fs", fold + 1, n_splits, time.time() - t_fold)
            if dev.type == "cuda":
                torch.cuda.empty_cache()
    else:
        for tr, te in gkf.split(df, y, groups=groups):
            tr_lines = df.iloc[tr]["line_id"].unique()
            pipe = fit_pca_sklearn_cpu(geno_wide, list(cols), tr_lines, pca_n, random_state)
            gpc_tr = map_pca_sklearn_to_rows(geno_wide, list(cols), df.iloc[tr]["line_id"], pipe).values
            gpc_te = map_pca_sklearn_to_rows(geno_wide, list(cols), df.iloc[te]["line_id"], pipe).values
            Xtr = np.hstack([gpc_tr, df.iloc[tr][nongeno_feature_cols].values.astype(float)])
            Xte = np.hstack([gpc_te, df.iloc[te][nongeno_feature_cols].values.astype(float)])
            model.fit(Xtr, y[tr])
            oof[te] = model.predict(Xte)
    return oof


def resolve_pca_device(cfg: Dict[str, Any]) -> tuple[bool, torch.device]:
    want = bool(cfg.get("use_gpu", True))
    if want and torch.cuda.is_available():
        d = int(cfg.get("cuda_device", 0))
        return True, torch.device(f"cuda:{d}")
    return False, torch.device("cpu")


def _gpc_corr_max_offdiag(gpc_df: pd.DataFrame, gpc_cols: List[str]) -> float:
    if len(gpc_cols) <= 1:
        return 0.0
    C = gpc_df[gpc_cols].corr(numeric_only=True).abs().values
    m = C.shape[0]
    tri = np.triu_indices(m, k=1)
    return float(C[tri].max())


def finalize_gpc_for_model(
    model_df: pd.DataFrame,
    geno: pd.DataFrame,
    snp_cols_pruned: List[str],
    X_snp_pruned: torch.Tensor,
    is_train: np.ndarray,
    pca_n: int,
    use_gpu: bool,
    device: torch.device,
    pca_once: bool,
    random_state: int,
    out_dir: Path,
) -> tuple[pd.DataFrame, List[str]]:
    tr_lines = model_df.loc[is_train, "line_id"].unique()
    k_req = int(pca_n)
    if use_gpu:
        if pca_once:
            X_fit_t = X_snp_pruned[is_train]
        else:
            X_fit_t = _unique_train_snp_tensor(geno, snp_cols_pruned, tr_lines, device)
        k_eff = _effective_pca_k(X_fit_t.shape[0], X_fit_t.shape[1], k_req)
        t0 = time.time()
        _sc, Vk, S, mean_f = pca_torch_gpu_fit(X_fit_t, k_eff)
        logging.info("Final GPU PCA fit: %.2fs | k=%s", time.time() - t0, k_eff)
        gpc_np = pca_torch_project(X_snp_pruned, mean_f, Vk).cpu().numpy()
        torch.save(
            {"mean": mean_f.cpu(), "V": Vk.cpu(), "S": S.cpu(), "snp_cols_pruned": snp_cols_pruned},
            out_dir / "gpu_pca_state.pt",
        )
        joblib.dump({"backend": "torch", "k": k_eff}, out_dir / "pca_pipeline.pkl")
        if device.type == "cuda":
            torch.cuda.empty_cache()
    else:
        pipe = fit_pca_sklearn_cpu(geno, snp_cols_pruned, tr_lines, k_req, random_state)
        joblib.dump(pipe, out_dir / "pca_pipeline.pkl")
        gpc_np = map_pca_sklearn_to_rows(geno, snp_cols_pruned, model_df["line_id"], pipe).values
    gpc_cols = [f"GPC_{i+1}" for i in range(gpc_np.shape[1])]
    gpc_df = pd.DataFrame(gpc_np, columns=gpc_cols)
    return gpc_df, gpc_cols


_cuda_ok = torch.cuda.is_available()
print("torch:", torch.__version__, "| CUDA:", _cuda_ok, torch.cuda.get_device_name(0) if _cuda_ok else "")


torch: 2.6.0+cu124 | CUDA: True NVIDIA GeForce RTX 4070 Laptop GPU


## CLI-Parity Entrypoint (Preserved)

The original script `main()` logic is preserved below with the same argparse default for `--config`.

In notebook mode, we execute the same flow in phased cells after this section.

In [8]:
def main() -> None:
    parser = argparse.ArgumentParser(description="Maize GxE YLDBE pipeline")
    parser.add_argument("--config", type=Path, default=Path("pipeline_config.yaml"))
    args = parser.parse_args()
    cfg_path = args.config.resolve()
    cfg = load_config(cfg_path)
    out = Path(cfg["output_dir"])
    if not out.is_absolute():
        out = (cfg_path.parent / out).resolve()
    setup_logging(out)
    rng = np.random.default_rng(int(cfg.get("random_state", 42)))

    cohort = str(cfg["cohort"]).upper()
    paths = cfg["paths"]
    years = cfg["years"]
    hypers = cfg["hypers"]

    pheno_path = Path(paths[f"pheno_{cohort.lower()}"])
    env_path = Path(paths["environmental"])
    imputed_dir = Path(paths[f"imputed_{cohort.lower()}_dir"])

    # Step 1
    pheno_env = step1_lock_analytical_unit(
        pheno_path, env_path, str(cfg.get("target_col", "YLD_BE")), out
    )

    # Step 2
    geno = step2_build_genotype_matrix(
        imputed_dir,
        cohort,
        out,
        cfg.get("genotype_max_snps"),
        cfg.get("genotype_max_pop_files"),
    )
    snp_cols = [c for c in geno.columns if c != "line_id"]

    # Step 3
    subset_pops = cfg.get("genotype_max_pop_files") is not None
    full, snp_cols, join_report = step3_validate_joins(
        pheno_env,
        geno,
        int(years["train_min"]),
        int(years["train_max"]),
        out,
        float(cfg.get("min_geno_match_rate", 0.7)),
        float(cfg.get("min_pheno_env_match_rate", 0.8)),
        float(cfg.get("nan_col_drop_frac", 0.5)),
        subset_populations=bool(subset_pops),
        save_full_dataset_csv=bool(cfg.get("save_full_dataset_csv", True)),
    )

    # Training / test masks
    tr_min, tr_max = int(years["train_min"]), int(years["train_max"])
    te_year = int(years["test_year"])
    train_year_mask = full["YEAR"].between(tr_min, tr_max) & full["YLDBE"].notna()
    test_year_mask = (full["YEAR"] == te_year) & full["YLDBE"].notna()
    model_df = full.loc[train_year_mask | test_year_mask].copy()
    model_df = model_df[model_df["line_id"].isin(set(geno["line_id"]))].copy()

    # Subsample training rows for runtime
    max_rows = cfg.get("max_train_rows")
    per_env = cfg.get("max_rows_per_env")
    train_only = model_df.loc[model_df["YEAR"].between(tr_min, tr_max)]
    if per_env:
        parts = []
        for env_id, chunk in train_only.groupby("env_id"):
            if len(chunk) > int(per_env):
                parts.append(chunk.sample(n=int(per_env), random_state=int(rng.integers(0, 1_000_000))))
            else:
                parts.append(chunk)
        train_only = pd.concat(parts, axis=0)
    if max_rows is not None and len(train_only) > int(max_rows):
        train_only = train_only.sample(n=int(max_rows), random_state=42)
    # Reattach test
    test_only = model_df.loc[model_df["YEAR"] == te_year]
    model_df = pd.concat([train_only, test_only], axis=0).drop_duplicates()
    snp_in_m = [c for c in snp_cols if c in model_df.columns]
    if snp_in_m:
        model_df = model_df.drop(columns=snp_in_m, errors="ignore")
        logging.info("Dropped %s raw SNP columns from modeling frame (kept separate geno matrix)", len(snp_in_m))

    logging.info("Modeling rows: %s (train-year subset + test year)", len(model_df))

    # Step 5 — env scaling (in-place on model_df slice columns that exist)
    model_df, env_numeric, env_scaler = step5_environment_features(model_df, str(cfg.get("env_scaling", "global")))
    joblib.dump(env_scaler, out / "env_scaler.pkl")

    # Step 4 + 6: GPU torch.pca_lowrank (or sklearn CPU) + variance-pruned SNPs
    pca_n = int(hypers.get("pca_components", 50))
    max_snps = int(cfg.get("max_snps", cfg.get("genotype_max_snps", 2000)))
    pca_once = bool(cfg.get("pca_once", False))
    use_gpu, device = resolve_pca_device(cfg)
    mem_pre = torch.cuda.memory_allocated() / 1e9 if use_gpu else 0.0
    t_pca0 = time.time()
    X_snp_pruned, snp_cols_pruned, _top_idx = build_X_snp_pruned_gpu(
        model_df, geno, snp_cols, max_snps, device
    )
    logging.info(
        "SNP tensor %s on %s | prep %.2fs | VRAM ~%.2f GB",
        tuple(X_snp_pruned.shape),
        device,
        time.time() - t_pca0,
        (torch.cuda.memory_allocated() / 1e9 - mem_pre) if use_gpu else 0.0,
    )

    is_train = model_df["YEAR"].between(tr_min, tr_max).values
    train_lines = model_df.loc[is_train, "line_id"].unique()
    Vk_frozen, mean_frozen = None, None
    if use_gpu and pca_once:
        X_fit_once = _unique_train_snp_tensor(geno, snp_cols_pruned, train_lines, device)
        k0 = _effective_pca_k(X_fit_once.shape[0], X_fit_once.shape[1], pca_n)
        _sc0, Vk_frozen, _S0, mean_frozen = pca_torch_gpu_fit(X_fit_once, k0)
        if device.type == "cuda":
            torch.cuda.empty_cache()

    gpc_df, gpc_cols = finalize_gpc_for_model(
        model_df,
        geno,
        snp_cols_pruned,
        X_snp_pruned,
        is_train,
        pca_n,
        use_gpu,
        device,
        pca_once,
        int(cfg.get("random_state", 42)),
        out,
    )
    model_df = pd.concat([model_df.reset_index(drop=True), gpc_df.reset_index(drop=True)], axis=1)

    gpc_df.to_csv(out / "gpu_gpc_components.csv", index=False)
    mx = _gpc_corr_max_offdiag(gpc_df, gpc_cols)
    logging.info("GPC max |corr| off-diag < 0.99: %s (max=%.4f)", mx < 0.99, mx)
    print("gpc collinear check (max abs corr off-diag < 0.99):", mx < 0.99, "| max =", mx)

    # Ridge baseline on pruned SNPs (train rows)
    ridge_X = (
        model_df.loc[is_train, ["line_id"]]
        .merge(geno, on="line_id", how="left")[snp_cols_pruned]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0.0)
        .values
    )
    ridge_y = model_df.loc[is_train, "YLDBE"].values
    ridge = Ridge(alpha=float(hypers.get("ridge_alpha", 1e-2)))
    ridge.fit(ridge_X, ridge_y)
    joblib.dump(ridge, out / "ridge_snp_baseline.pkl")
    logging.info("Ridge SNP baseline fit on %s train rows", len(ridge_y))

    # Step 6 env + GxE (pairs chosen on train slice; columns on full model_df)
    env_only_numeric = [c for c in env_numeric if c in model_df.columns and not str(c).startswith("GPC_")]
    env_chosen, gxe_pairs = step6_select_env_and_gxe(
        model_df.loc[is_train].copy(),
        env_only_numeric,
        gpc_cols,
        rng,
        int(cfg.get("max_env_features_gxe", 20)),
        int(cfg.get("n_interaction_pairs", 10)),
    )
    interact_cols: List[str] = []
    gxe_parts: Dict[str, pd.Series] = {}
    for gc, ec in gxe_pairs:
        cname = f"GXE__{gc}__x__{ec}"
        gxe_parts[cname] = model_df[gc].astype(float) * model_df[ec].astype(float)
        interact_cols.append(cname)
    if gxe_parts:
        model_df = pd.concat([model_df, pd.DataFrame(gxe_parts)], axis=1)

    past_cols = [c for c in ("past_yld_mean", "past_yld_median") if c in model_df.columns]
    nongeno_feature_cols = env_chosen + past_cols + interact_cols
    nongeno_feature_cols = [c for c in nongeno_feature_cols if c in model_df.columns]
    pd.Series(nongeno_feature_cols, name="env_and_gxe_column").to_csv(
        out / "env_gxe_feature_columns.csv", index=False
    )
    feature_cols = gpc_cols + nongeno_feature_cols
    G = model_df[gpc_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).values
    Ng = model_df[nongeno_feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).values
    X_all = np.hstack([G, Ng])
    y_all = model_df["YLDBE"].values.astype(float)
    groups_all = model_df["env_id"].astype(str).values
    train_idx = is_train

    # Phase A — RandomizedSearchCV on train rows only
    n_train_rows = int(train_idx.sum())
    use_gpu_cfg = bool(cfg.get("use_gpu", False))
    cuda_device = int(cfg.get("cuda_device", 0))
    xgb_ok = False
    torch_cuda = False
    xgb = None
    if use_gpu_cfg:
        try:
            import xgboost as xgb  # type: ignore

            xgb_ok = True
        except Exception:
            logging.warning("xgboost import failed; Step 7a will use sklearn GBR CPU fallback.")
        try:
            import torch  # type: ignore

            torch_cuda = bool(torch.cuda.is_available())
        except Exception:
            torch_cuda = False
    use_gpu_xgb = bool(use_gpu_cfg and xgb_ok and torch_cuda)

    n_iter = int(hypers.get("random_search_iter", 50))
    if n_train_rows < 15_000:
        n_iter = min(n_iter, 12)

    gb_hyp = hypers.get("gbr", {})
    fixed_params_gbr = {
        "n_estimators": int(gb_hyp.get("n_estimators", 500)),
        "max_depth": int(gb_hyp.get("max_depth", 6)),
        "learning_rate": float(gb_hyp.get("learning_rate", 0.05)),
        "subsample": float(gb_hyp.get("subsample", 0.8)),
    }
    skip_search = bool(cfg.get("skip_random_search", False))

    if use_gpu_xgb:
        log_step("Step 7a: XGBoost (GPU) + RandomizedSearchCV (GroupKFold)")
        logging.info(
            "Step 7a GPU enabled (use_gpu=%s, torch_cuda=%s, device=cuda:%s).",
            use_gpu_cfg,
            torch_cuda,
            cuda_device,
        )
        if n_train_rows < 15_000:
            param_dist = {
                "n_estimators": [100, 200, 300],
                "max_depth": [4, 5, 6],
                "learning_rate": [0.05, 0.08],
                "subsample": [0.8],
                "colsample_bytree": [0.8, 1.0],
            }
        else:
            param_dist = {
                "n_estimators": [300, 500, 700],
                "max_depth": [4, 6, 8],
                "learning_rate": [0.03, 0.05, 0.08],
                "subsample": [0.75, 0.8, 0.9],
                "colsample_bytree": [0.6, 0.8, 1.0],
            }

        fixed_params_xgb = {
            "n_estimators": int(gb_hyp.get("n_estimators", 500)),
            "max_depth": int(gb_hyp.get("max_depth", 6)),
            "learning_rate": float(gb_hyp.get("learning_rate", 0.05)),
            "subsample": float(gb_hyp.get("subsample", 0.8)),
            "colsample_bytree": 0.8,
            "objective": "reg:squarederror",
            "tree_method": "hist",
            "device": f"cuda:{cuda_device}",
            "predictor": "gpu_predictor",
            "n_jobs": 1,
            "random_state": 42,
            "verbosity": 0,
        }

        if skip_search:
            logging.info(
                "Skipping RandomizedSearchCV (small train or skip_random_search); using config xgboost params %s",
                fixed_params_xgb,
            )
            search = type(
                "SearchStub",
                (),
                {"best_params_": fixed_params_xgb, "best_estimator_": xgb.XGBRegressor(**fixed_params_xgb)},
            )()
        else:
            search = RandomizedSearchCV(
                xgb.XGBRegressor(**fixed_params_xgb),
                param_dist,
                n_iter=n_iter,
                cv=GroupKFold(n_splits=int(hypers.get("group_cv_splits", 5))),
                random_state=42,
                n_jobs=1,
                scoring="neg_root_mean_squared_error",
            )
            search.fit(X_all[train_idx], y_all[train_idx], groups=groups_all[train_idx])
            logging.info("Best XGBoost GPU params: %s", search.best_params_)
        oof_model = xgb.XGBRegressor(**search.best_params_)
    else:
        if use_gpu_cfg and not use_gpu_xgb:
            logging.warning(
                "use_gpu=true but Step 7a GPU path unavailable (xgboost=%s, torch_cuda=%s); using sklearn GBR CPU fallback.",
                xgb_ok,
                torch_cuda,
            )
        log_step("Step 7a: GradientBoosting + RandomizedSearchCV (GroupKFold)")
        import xgboost as xgb
        GPU_REQUIRED_ERR = "GPU XGBoost required for hackathon speed – check CUDA install & VRAM. CPU too slow for SNPs."
        if not bool(cfg.get("use_gpu", True)):
            raise RuntimeError(GPU_REQUIRED_ERR)
        if not bool(torch is not None and torch.cuda.is_available()):
            raise RuntimeError(GPU_REQUIRED_ERR)
        gbr = xgb.XGBRegressor(
            objective="reg:squarederror",
            tree_method="hist",
            device=f"cuda:{int(cfg.get('cuda_device', 0))}",
            predictor="gpu_predictor",
            random_state=42,
            n_jobs=1,
            verbosity=0,
        )
        if n_train_rows < 15_000:
            param_dist = {
                "n_estimators": [100, 200, 300],
                "max_depth": [4, 5, 6],
                "learning_rate": [0.05, 0.08],
                "subsample": [0.8],
            }
        else:
            param_dist = {
                "n_estimators": [300, 500, 700],
                "max_depth": [4, 6, 8],
                "learning_rate": [0.03, 0.05, 0.08],
                "subsample": [0.75, 0.8, 0.9],
            }

        if skip_search:
            logging.info(
                "Skipping RandomizedSearchCV (small train or skip_random_search); using config gbr params %s",
                fixed_params_gbr,
            )
            search = type(
                "SearchStub",
                (),
                {
                    "best_params_": fixed_params_gbr,
                    "best_estimator_": xgb.XGBRegressor(
                        **fixed_params_gbr,
                        objective="reg:squarederror",
                        tree_method="hist",
                        device=f"cuda:{int(cfg.get('cuda_device', 0))}",
                        predictor="gpu_predictor",
                        random_state=42,
                        n_jobs=1,
                        verbosity=0,
                    ),
                },
            )()
        else:
            search = RandomizedSearchCV(
                gbr,
                param_dist,
                n_iter=n_iter,
                cv=GroupKFold(n_splits=int(hypers.get("group_cv_splits", 5))),
                random_state=42,
                n_jobs=int(cfg.get("sklearn_n_jobs", 1)),
                scoring="neg_root_mean_squared_error",
            )
            search.fit(X_all[train_idx], y_all[train_idx], groups=groups_all[train_idx])
            logging.info("Best GBR params: %s", search.best_params_)
        oof_model = xgb.XGBRegressor(
            **search.best_params_,
            objective="reg:squarederror",
            tree_method="hist",
            device=f"cuda:{int(cfg.get('cuda_device', 0))}",
            predictor="gpu_predictor",
            random_state=42,
            n_jobs=1,
            verbosity=0,
        )

    # OOF on train with PCA refit per fold (Step 4 + 8)
    log_step("Step 8: OOF predictions (PCA refit per fold)")
    n_folds_oof = int(cfg.get("gpu_pca_n_folds", hypers.get("group_cv_splits", 5)))
    tr_pos = np.flatnonzero(train_idx)
    oof_train = grouped_oof_with_pca(
        model_df.loc[train_idx].reset_index(drop=True),
        snp_cols,
        geno,
        nongeno_feature_cols,
        y_all[train_idx],
        groups_all[train_idx],
        n_folds_oof,
        pca_n,
        oof_model,
        int(cfg.get("random_state", 42)),
        snp_cols_pruned=snp_cols_pruned,
        X_snp_pruned_gpu=X_snp_pruned,
        model_row_positions=tr_pos,
        use_gpu=use_gpu,
        pca_once=pca_once,
        Vk_frozen=Vk_frozen,
        mean_frozen=mean_frozen,
    )
    oof_full = np.full(len(y_all), np.nan)
    oof_full[np.where(train_idx)[0]] = oof_train
    model_df["oof_pred"] = oof_full

    glob_metrics = metrics_dict(y_all[train_idx], oof_train)
    logging.info("OOF train metrics: %s", glob_metrics)

    by_env_rows = []
    for eid, chunk in model_df.loc[train_idx].groupby("env_id"):
        m = metrics_dict(chunk["YLDBE"].values, chunk["oof_pred"].values)
        m["env_id"] = eid
        m["n"] = len(chunk)
        by_env_rows.append(m)
    by_env = pd.DataFrame(by_env_rows)
    by_env.to_csv(out / "metrics_by_env_oof.csv", index=False)

    fig, ax = plt.subplots(figsize=(8, 5))
    tr = model_df.loc[train_idx]
    res = tr["YLDBE"] - tr["oof_pred"]
    ax.axhline(0, color="gray", lw=0.8)
    ax.scatter(tr["oof_pred"], res, alpha=0.15, s=8)
    ax.set_xlabel("OOF predicted YLDBE")
    ax.set_ylabel("Residual (obs - pred)")
    ax.set_title("Residuals vs OOF prediction (train years)")
    fig.tight_layout()
    fig.savefig(out / "res_plot.png", dpi=150)
    plt.close(fig)

    # Phase B — HistGradientBoosting or XGB with test-year early stopping
    log_step("Step 7b: Phase B booster + env/year holdout early stopping")
    X_tr = X_all[train_idx]
    y_tr = y_all[train_idx]
    te_idx = model_df["YEAR"].values == te_year
    X_va = X_all[te_idx]
    y_va = y_all[te_idx]
    hgb: Any
    if cfg.get("use_xgboost", False):
        try:
            import xgboost as xgb

            esr = int(hypers.get("early_stopping_rounds", 50))
            hgb = xgb.XGBRegressor(
                n_estimators=int(hypers["hist_gb"]["max_iter"]),
                max_depth=int(hypers["hist_gb"]["max_depth"]),
                learning_rate=float(hypers["hist_gb"]["learning_rate"]),
                subsample=0.8,
                random_state=42,
                n_jobs=-1,
            )
            if len(X_va) >= 20:
                hgb.fit(
                    X_tr,
                    y_tr,
                    eval_set=[(X_va, y_va)],
                    verbose=False,
                    early_stopping_rounds=esr,
                )
            else:
                hgb.fit(X_tr, y_tr)
        except ImportError as e:
            raise RuntimeError("GPU XGBoost required for hackathon speed – check CUDA install & VRAM. CPU too slow for SNPs.") from e
    joblib.dump(hgb, out / "best_model.pkl")

    model_df["pred_final"] = hgb.predict(X_all)
    oof_path = out / "oof_preds.csv"
    model_df[["line_id", "env_id", "YEAR", "LOC", "YLDBE", "oof_pred", "pred_final"]].to_csv(
        oof_path, index=False
    )

    # Step 9 ranking
    log_step("Step 9: Top-k per environment")
    topk = int(cfg.get("topk", 10))
    rank_rows = []
    for env_id, chunk in model_df.loc[te_idx].groupby("env_id"):
        sub = chunk.sort_values("pred_final", ascending=False).head(topk).copy()
        sub["rank"] = np.arange(1, len(sub) + 1)
        sub["uncertainty_oof_rmse"] = glob_metrics["rmse"]
        rank_rows.append(sub)
    rankings = pd.concat(rank_rows, axis=0) if rank_rows else pd.DataFrame()
    rankings.to_csv(out / "rankings_topk_per_env.csv", index=False)

    # Step 10 interpretation
    log_step("Step 10: Permutation importance + buckets")
    perm = permutation_importance(
        hgb,
        X_tr,
        y_tr,
        n_repeats=10,
        random_state=42,
        n_jobs=int(cfg.get("sklearn_n_jobs", 1)),
    )
    fi = pd.DataFrame({"feature": feature_cols, "importance_mean": perm.importances_mean, "importance_std": perm.importances_std})
    fi = fi.sort_values("importance_mean", ascending=False)
    fi.to_csv(out / "perm_importance.csv", index=False)

    def bucket(name: str) -> str:
        if name.startswith("GPC_"):
            return "Genetic_PCA"
        if name.startswith("GXE__"):
            return "GxE_interaction"
        if any(x in name.upper() for x in ("PRCP", "TAVG", "HTDD", "CLDD", "DP01", "DP10")):
            return "Weather"
        if any(x in name.upper() for x in ("CLAY", "SILT", "SAND", "PH", "SOC", "NITROGEN", "CFVO")):
            return "Soil"
        if "past_yld" in name:
            return "Past_pheno"
        return "Other"

    fi["bucket"] = fi["feature"].map(bucket)
    bucketed = fi.groupby("bucket")["importance_mean"].sum().reset_index()
    bucketed.to_csv(out / "perm_importance_buckets.csv", index=False)
    fi.head(20).to_csv(out / "interpretation_top_features_slide.csv", index=False)

    if cfg.get("use_shap", True):
        try:
            import shap

            explainer = shap.Explainer(hgb.predict, X_tr[: min(2000, len(X_tr))])
            sv = explainer(X_tr[: min(2000, len(X_tr))])
            shap.summary_plot(sv, feature_names=feature_cols, show=False)
            plt.tight_layout()
            plt.savefig(out / "shap.png", dpi=150, bbox_inches="tight")
            plt.close()
        except Exception as e:
            logging.warning("SHAP skipped: %s", e)

    # Step 11 summary
    summary = {
        "cohort": cohort,
        "join_report": join_report,
        "modeling_rows": int(len(model_df)),
        "oof_global_metrics": glob_metrics,
        "best_gbr_cv_params": search.best_params_,
        "feature_columns": feature_cols,
        "paths": {k: str(Path(v)) for k, v in paths.items()},
    }

    def _json_default(o: Any) -> Any:
        if isinstance(o, (np.floating, np.integer)):
            return o.item()
        return str(o)

    with open(out / "run_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, default=_json_default)
    logging.info("Done. Artifacts in %s", out)

# Original script entrypoint (kept for reference):
# if __name__ == "__main__":
#     main()


## Notebook Configuration (Argparse Replacement)

Notebook-friendly replacement for CLI args. Keep defaults aligned with script behavior, but editable here.

- Script CLI default: `--config pipeline_config.yaml`
- Notebook default below uses the same config file name in the working directory.

In [9]:
from types import SimpleNamespace

args = SimpleNamespace(
    config=Path("pipeline_config.yaml"),
)

cfg_path = args.config.resolve()
cfg = load_config(cfg_path)
out = Path(cfg["output_dir"])
if not out.is_absolute():
    out = (cfg_path.parent / out).resolve()
setup_logging(out)

rng = np.random.default_rng(int(cfg.get("random_state", 42)))

cohort = str(cfg["cohort"]).upper()
paths = cfg["paths"]
years = cfg["years"]
hypers = cfg["hypers"]

pheno_path = Path(paths[f"pheno_{cohort.lower()}"])
env_path = Path(paths["environmental"])
imputed_dir = Path(paths[f"imputed_{cohort.lower()}_dir"])

print("Config:", cfg_path)
print("Output:", out)
print("Cohort:", cohort)


Config: C:\Users\kensm\Gods-Mercy\pipeline_config.yaml
Output: C:\Users\kensm\Gods-Mercy\output
Cohort: C1


## Execute Pipeline (Phased)

Run the following cells in order.

In [10]:
# Step 1
pheno_env = step1_lock_analytical_unit(
    pheno_path, env_path, str(cfg.get("target_col", "YLD_BE")), out
)

# Step 2
geno = step2_build_genotype_matrix(
    imputed_dir,
    cohort,
    out,
    cfg.get("genotype_max_snps"),
    cfg.get("genotype_max_pop_files"),
)
snp_cols = [c for c in geno.columns if c != "line_id"]

# Step 3
subset_pops = cfg.get("genotype_max_pop_files") is not None
full, snp_cols, join_report = step3_validate_joins(
    pheno_env,
    geno,
    int(years["train_min"]),
    int(years["train_max"]),
    out,
    float(cfg.get("min_geno_match_rate", 0.7)),
    float(cfg.get("min_pheno_env_match_rate", 0.8)),
    float(cfg.get("nan_col_drop_frac", 0.5)),
    subset_populations=bool(subset_pops),
    save_full_dataset_csv=bool(cfg.get("save_full_dataset_csv", True)),
)

print("Join report:")
print(join_report)


2026-03-28 17:50:06,131 [INFO] === Step 1: Lock analytical unit ===
2026-03-28 17:50:08,624 [WARNING] Dropped 4488 phenotype rows with non-parseable line_id
2026-03-28 17:50:09,136 [WARNING] Duplicate grain rows: 2924 - dropping duplicates (keep first)
2026-03-28 17:50:31,314 [INFO] Saved 530986 rows to C:\Users\kensm\Gods-Mercy\output\pheno_env.csv
2026-03-28 17:50:31,347 [INFO] === Step 2: Build genotype matrix ===
2026-03-28 17:50:34,423 [INFO] Loaded 50/499 population files
2026-03-28 17:50:37,327 [INFO] Loaded 100/499 population files
2026-03-28 17:50:40,437 [INFO] Loaded 150/499 population files
2026-03-28 17:50:43,379 [INFO] Loaded 200/499 population files
2026-03-28 17:50:46,603 [INFO] Loaded 250/499 population files
2026-03-28 17:50:49,866 [INFO] Loaded 300/499 population files
2026-03-28 17:50:52,859 [INFO] Loaded 350/499 population files
2026-03-28 17:50:55,475 [INFO] Loaded 400/499 population files
2026-03-28 17:50:57,962 [INFO] Loaded 450/499 population files
2026-03-28 17

In [11]:
# Training / test masks
tr_min, tr_max = int(years["train_min"]), int(years["train_max"])
te_year = int(years["test_year"])
train_year_mask = full["YEAR"].between(tr_min, tr_max) & full["YLDBE"].notna()
test_year_mask = (full["YEAR"] == te_year) & full["YLDBE"].notna()
model_df = full.loc[train_year_mask | test_year_mask].copy()
model_df = model_df[model_df["line_id"].isin(set(geno["line_id"]))].copy()

# Subsample training rows for runtime
max_rows = cfg.get("max_train_rows")
per_env = cfg.get("max_rows_per_env")
train_only = model_df.loc[model_df["YEAR"].between(tr_min, tr_max)]
if per_env:
    parts = []
    for env_id, chunk in train_only.groupby("env_id"):
        if len(chunk) > int(per_env):
            parts.append(chunk.sample(n=int(per_env), random_state=int(rng.integers(0, 1_000_000))))
        else:
            parts.append(chunk)
    train_only = pd.concat(parts, axis=0)
if max_rows is not None and len(train_only) > int(max_rows):
    train_only = train_only.sample(n=int(max_rows), random_state=42)

# Reattach test
test_only = model_df.loc[model_df["YEAR"] == te_year]
model_df = pd.concat([train_only, test_only], axis=0).drop_duplicates()

snp_in_m = [c for c in snp_cols if c in model_df.columns]
if snp_in_m:
    model_df = model_df.drop(columns=snp_in_m, errors="ignore")
    logging.info("Dropped %s raw SNP columns from modeling frame (kept separate geno matrix)", len(snp_in_m))

logging.info("Modeling rows: %s (train-year subset + test year)", len(model_df))

# Step 5 — env scaling
model_df, env_numeric, env_scaler = step5_environment_features(model_df, str(cfg.get("env_scaling", "global")))
joblib.dump(env_scaler, out / "env_scaler.pkl")

# Step 4 + 6: GPU PCA (torch.pca_lowrank) or sklearn fallback
pca_n = int(hypers.get("pca_components", 50))
max_snps = int(cfg.get("max_snps", cfg.get("genotype_max_snps", 2000)))
pca_once = bool(cfg.get("pca_once", False))
use_gpu, device = resolve_pca_device(cfg)
mem_pre = torch.cuda.memory_allocated() / 1e9 if use_gpu else 0.0
t_pca0 = time.time()
X_snp_pruned, snp_cols_pruned, _top_idx = build_X_snp_pruned_gpu(
    model_df, geno, snp_cols, max_snps, device
)
logging.info(
    "SNP tensor %s on %s | prep %.2fs",
    tuple(X_snp_pruned.shape),
    device,
    time.time() - t_pca0,
)

is_train = model_df["YEAR"].between(tr_min, tr_max).values
train_lines = model_df.loc[is_train, "line_id"].unique()
Vk_frozen, mean_frozen = None, None
if use_gpu and pca_once:
    X_fit_once = _unique_train_snp_tensor(geno, snp_cols_pruned, train_lines, device)
    k0 = _effective_pca_k(X_fit_once.shape[0], X_fit_once.shape[1], pca_n)
    _sc0, Vk_frozen, _S0, mean_frozen = pca_torch_gpu_fit(X_fit_once, k0)
    if device.type == "cuda":
        torch.cuda.empty_cache()

gpc_df, gpc_cols = finalize_gpc_for_model(
    model_df,
    geno,
    snp_cols_pruned,
    X_snp_pruned,
    is_train,
    pca_n,
    use_gpu,
    device,
    pca_once,
    int(cfg.get("random_state", 42)),
    out,
)
model_df = pd.concat([model_df.reset_index(drop=True), gpc_df.reset_index(drop=True)], axis=1)

gpc_df.to_csv(out / "gpu_gpc_components.csv", index=False)
mx = _gpc_corr_max_offdiag(gpc_df, gpc_cols)
print("gpc collinear check (max abs corr off-diag < 0.99):", mx < 0.99, "| max =", mx)

# Ridge baseline on pruned SNPs (train rows)
ridge_X = (
    model_df.loc[is_train, ["line_id"]]
    .merge(geno, on="line_id", how="left")[snp_cols_pruned]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0.0)
    .values
)
ridge_y = model_df.loc[is_train, "YLDBE"].values
ridge = Ridge(alpha=float(hypers.get("ridge_alpha", 1e-2)))
ridge.fit(ridge_X, ridge_y)
joblib.dump(ridge, out / "ridge_snp_baseline.pkl")
logging.info("Ridge SNP baseline fit on %s train rows", len(ridge_y))

# Step 6 env + GxE
env_only_numeric = [c for c in env_numeric if c in model_df.columns and not str(c).startswith("GPC_")]
env_chosen, gxe_pairs = step6_select_env_and_gxe(
    model_df.loc[is_train].copy(),
    env_only_numeric,
    gpc_cols,
    rng,
    int(cfg.get("max_env_features_gxe", 20)),
    int(cfg.get("n_interaction_pairs", 10)),
)

interact_cols: List[str] = []
gxe_parts: Dict[str, pd.Series] = {}
for gc, ec in gxe_pairs:
    cname = f"GXE__{gc}__x__{ec}"
    gxe_parts[cname] = model_df[gc].astype(float) * model_df[ec].astype(float)
    interact_cols.append(cname)
if gxe_parts:
    model_df = pd.concat([model_df, pd.DataFrame(gxe_parts)], axis=1)

past_cols = [c for c in ("past_yld_mean", "past_yld_median") if c in model_df.columns]
nongeno_feature_cols = env_chosen + past_cols + interact_cols
nongeno_feature_cols = [c for c in nongeno_feature_cols if c in model_df.columns]

pd.Series(nongeno_feature_cols, name="env_and_gxe_column").to_csv(
    out / "env_gxe_feature_columns.csv", index=False
)

feature_cols = gpc_cols + nongeno_feature_cols
G = model_df[gpc_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).values
Ng = model_df[nongeno_feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).values
X_all = np.hstack([G, Ng])
y_all = model_df["YLDBE"].values.astype(float)
groups_all = model_df["env_id"].astype(str).values
train_idx = is_train

print("Feature matrix shape:", X_all.shape)
print("Train rows:", int(train_idx.sum()))


2026-03-28 17:54:22,774 [INFO] Modeling rows: 82522 (train-year subset + test year)
2026-03-28 17:54:22,775 [INFO] === Step 5: Environment covariates ===
2026-03-28 17:54:26,941 [INFO] GPU SNPs pruned: 2000 (from 2500)
2026-03-28 17:54:26,967 [INFO] SNP tensor (82522, 2000) on cuda:0 | prep 3.59s
2026-03-28 17:54:28,728 [INFO] Final GPU PCA fit: 0.27s | k=50
gpc collinear check (max abs corr off-diag < 0.99): True | max = 0.25543080069951296
2026-03-28 17:56:23,646 [INFO] Ridge SNP baseline fit on 45000 train rows
2026-03-28 17:56:23,691 [INFO] === Step 6: GxE feature refinement ===


c:\Users\kensm\anaconda3\Lib\site-packages\sklearn\linear_model\_ridge.py:215: LinAlgWarning: Ill-conditioned matrix (rcond=6.85439e-10): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


Feature matrix shape: (82522, 82)
Train rows: 45000


In [14]:
# Phase A — RandomizedSearchCV on train rows only
n_train_rows = int(train_idx.sum())
n_iter = int(hypers.get("random_search_iter", 50))
if n_train_rows < 15_000:
    n_iter = min(n_iter, 12)

gb_hyp = hypers.get("gbr", {})
fixed_params = {
    "n_estimators": int(gb_hyp.get("n_estimators", 500)),
    "max_depth": int(gb_hyp.get("max_depth", 6)),
    "learning_rate": float(gb_hyp.get("learning_rate", 0.05)),
    "subsample": float(gb_hyp.get("subsample", 0.8)),
}

skip_search = bool(cfg.get("skip_random_search", False))
use_gpu_cfg = bool(cfg.get("use_gpu", False))
cuda_device = int(cfg.get("cuda_device", 0))
xgb_ok = False
torch_cuda = False
xgb = None

if use_gpu_cfg:
    try:
        import xgboost as xgb

        xgb_ok = True
    except Exception as first_err:
        logging.warning("xgboost import failed initially (%s). Trying pip install...", first_err)
        try:
            import subprocess
            import sys

            subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "xgboost"])
            import xgboost as xgb

            xgb_ok = True
            logging.info("xgboost installed successfully in current environment.")
        except Exception as install_err:
            logging.warning("xgboost install/import failed; Step 7a will use sklearn GBR CPU fallback. Error: %s", install_err)
            xgb_ok = False

    try:
        import torch

        torch_cuda = bool(torch.cuda.is_available())
    except Exception:
        torch_cuda = False

use_gpu_xgb = bool(use_gpu_cfg and xgb_ok and torch_cuda)

if use_gpu_xgb:
    log_step("Step 7a: XGBoost (GPU) + RandomizedSearchCV (GroupKFold)")
    logging.info(
        "Step 7a GPU enabled (use_gpu=%s, torch_cuda=%s, device=cuda:%s).",
        use_gpu_cfg,
        torch_cuda,
        cuda_device,
    )
    if n_train_rows < 15_000:
        param_dist = {
            "n_estimators": [100, 200, 300],
            "max_depth": [4, 5, 6],
            "learning_rate": [0.05, 0.08],
            "subsample": [0.8],
            "colsample_bytree": [0.8, 1.0],
        }
    else:
        param_dist = {
            "n_estimators": [300, 500, 700],
            "max_depth": [4, 6, 8],
            "learning_rate": [0.03, 0.05, 0.08],
            "subsample": [0.75, 0.8, 0.9],
            "colsample_bytree": [0.6, 0.8, 1.0],
        }

    fixed_params_xgb = {
        "n_estimators": int(gb_hyp.get("n_estimators", 500)),
        "max_depth": int(gb_hyp.get("max_depth", 6)),
        "learning_rate": float(gb_hyp.get("learning_rate", 0.05)),
        "subsample": float(gb_hyp.get("subsample", 0.8)),
        "colsample_bytree": 0.8,
        "objective": "reg:squarederror",
        "tree_method": "hist",
        "device": f"cuda:{cuda_device}",
        "predictor": "gpu_predictor",
        "n_jobs": 1,
        "random_state": 42,
        "verbosity": 0,
    }

    if skip_search:
        logging.info(
            "Skipping RandomizedSearchCV (small train or skip_random_search); using config xgboost params %s",
            fixed_params_xgb,
        )
        search = type(
            "SearchStub",
            (),
            {"best_params_": fixed_params_xgb, "best_estimator_": xgb.XGBRegressor(**fixed_params_xgb)},
        )()
    else:
        search = RandomizedSearchCV(
            xgb.XGBRegressor(**fixed_params_xgb),
            param_dist,
            n_iter=n_iter,
            cv=GroupKFold(n_splits=int(hypers.get("group_cv_splits", 5))),
            random_state=42,
            n_jobs=1,
            scoring="neg_root_mean_squared_error",
        )
        search.fit(X_all[train_idx], y_all[train_idx], groups=groups_all[train_idx])
        logging.info("Best XGBoost GPU params: %s", search.best_params_)

    oof_model = xgb.XGBRegressor(**search.best_params_)
else:
    if use_gpu_cfg and not use_gpu_xgb:
        logging.warning(
            "use_gpu=true but Step 7a GPU path unavailable (xgboost=%s, torch_cuda=%s); using sklearn GBR CPU fallback.",
            xgb_ok,
            torch_cuda,
        )

    log_step("Step 7a: GradientBoosting + RandomizedSearchCV (GroupKFold)")
    gbr = GradientBoostingRegressor(random_state=42)

    if n_train_rows < 15_000:
        param_dist = {
            "n_estimators": [100, 200, 300],
            "max_depth": [4, 5, 6],
            "learning_rate": [0.05, 0.08],
            "subsample": [0.8],
        }
    else:
        param_dist = {
            "n_estimators": [300, 500, 700],
            "max_depth": [4, 6, 8],
            "learning_rate": [0.03, 0.05, 0.08],
            "subsample": [0.75, 0.8, 0.9],
        }

    if skip_search:
        logging.info(
            "Skipping RandomizedSearchCV (small train or skip_random_search); using config gbr params %s",
            fixed_params,
        )
        search = type(
            "SearchStub",
            (),
            {"best_params_": fixed_params, "best_estimator_": GradientBoostingRegressor(random_state=42, **fixed_params)},
        )()
    else:
        search = RandomizedSearchCV(
            gbr,
            param_dist,
            n_iter=n_iter,
            cv=GroupKFold(n_splits=int(hypers.get("group_cv_splits", 5))),
            random_state=42,
            n_jobs=int(cfg.get("sklearn_n_jobs", 1)),
            scoring="neg_root_mean_squared_error",
        )
        search.fit(X_all[train_idx], y_all[train_idx], groups=groups_all[train_idx])
        logging.info("Best GBR params: %s", search.best_params_)

    oof_model = GradientBoostingRegressor(**search.best_params_, random_state=42)

# OOF on train with PCA refit per fold (Step 4 + 8)
log_step("Step 8: OOF predictions (PCA refit per fold)")
n_folds_oof = int(cfg.get("gpu_pca_n_folds", hypers.get("group_cv_splits", 5)))
tr_pos = np.flatnonzero(train_idx)
oof_train = grouped_oof_with_pca(
    model_df.loc[train_idx].reset_index(drop=True),
    snp_cols,
    geno,
    nongeno_feature_cols,
    y_all[train_idx],
    groups_all[train_idx],
    n_folds_oof,
    pca_n,
    oof_model,
    int(cfg.get("random_state", 42)),
    snp_cols_pruned=snp_cols_pruned,
    X_snp_pruned_gpu=X_snp_pruned,
    model_row_positions=tr_pos,
    use_gpu=use_gpu,
    pca_once=pca_once,
    Vk_frozen=Vk_frozen,
    mean_frozen=mean_frozen,
)

oof_full = np.full(len(y_all), np.nan)
oof_full[np.where(train_idx)[0]] = oof_train
model_df["oof_pred"] = oof_full

glob_metrics = metrics_dict(y_all[train_idx], oof_train)
logging.info("OOF train metrics: %s", glob_metrics)

by_env_rows = []
for eid, chunk in model_df.loc[train_idx].groupby("env_id"):
    m = metrics_dict(chunk["YLDBE"].values, chunk["oof_pred"].values)
    m["env_id"] = eid
    m["n"] = len(chunk)
    by_env_rows.append(m)
by_env = pd.DataFrame(by_env_rows)
by_env.to_csv(out / "metrics_by_env_oof.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 5))
tr = model_df.loc[train_idx]
res = tr["YLDBE"] - tr["oof_pred"]
ax.axhline(0, color="gray", lw=0.8)
ax.scatter(tr["oof_pred"], res, alpha=0.15, s=8)
ax.set_xlabel("OOF predicted YLDBE")
ax.set_ylabel("Residual (obs - pred)")
ax.set_title("Residuals vs OOF prediction (train years)")
fig.tight_layout()
fig.savefig(out / "res_plot.png", dpi=150)
plt.close(fig)

glob_metrics


2026-03-28 18:16:41,920 [WARNING] xgboost import failed initially (No module named 'xgboost'). Trying pip install...
2026-03-28 18:16:59,992 [INFO] xgboost installed successfully in current environment.
2026-03-28 18:16:59,994 [INFO] === Step 7a: XGBoost (GPU) + RandomizedSearchCV (GroupKFold) ===
2026-03-28 18:16:59,995 [INFO] Step 7a GPU enabled (use_gpu=True, torch_cuda=True, device=cuda:0).
2026-03-28 18:38:19,764 [INFO] Best XGBoost GPU params: {'subsample': 0.9, 'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.03, 'colsample_bytree': 1.0}
2026-03-28 18:38:19,765 [INFO] === Step 8: OOF predictions (PCA refit per fold) ===
2026-03-28 18:38:23,411 [INFO] GPU PCA OOF fold 1/3 in 3.52s
2026-03-28 18:38:26,461 [INFO] GPU PCA OOF fold 2/3 in 3.05s
2026-03-28 18:38:29,545 [INFO] GPU PCA OOF fold 3/3 in 3.08s
2026-03-28 18:38:29,565 [INFO] OOF train metrics: {'rmse': 0.8791358870895697, 'mae': 0.2509359101207139, 'r2': 0.9994152865111873}


{'rmse': 0.8791358870895697,
 'mae': 0.2509359101207139,
 'r2': 0.9994152865111873}

In [ ]:
# Phase B — HistGradientBoosting or optional XGBoost with early stopping
log_step("Step 7b: Phase B booster + env/year holdout early stopping")
X_tr = X_all[train_idx]
y_tr = y_all[train_idx]
te_idx = model_df["YEAR"].values == te_year
X_va = X_all[te_idx]
y_va = y_all[te_idx]

hgb: Any
if cfg.get("use_xgboost", False):
    try:
        import xgboost as xgb

        esr = int(hypers.get("early_stopping_rounds", 50))
        hgb = xgb.XGBRegressor(
            n_estimators=int(hypers["hist_gb"]["max_iter"]),
            max_depth=int(hypers["hist_gb"]["max_depth"]),
            learning_rate=float(hypers["hist_gb"]["learning_rate"]),
            subsample=0.8,
            random_state=42,
            n_jobs=-1,
        )
        if len(X_va) >= 20:
            hgb.fit(
                X_tr,
                y_tr,
                eval_set=[(X_va, y_va)],
                verbose=False,
                early_stopping_rounds=esr,
            )
        else:
            hgb.fit(X_tr, y_tr)
    except ImportError:
        logging.warning("xgboost not installed; falling back to HistGradientBoostingRegressor")
        cfg["use_xgboost"] = False

if not cfg.get("use_xgboost", False):
    hgb = HistGradientBoostingRegressor(
        max_iter=int(hypers["hist_gb"]["max_iter"]),
        max_depth=int(hypers["hist_gb"]["max_depth"]),
        learning_rate=float(hypers["hist_gb"]["learning_rate"]),
        random_state=42,
        early_stopping=True,
        validation_fraction=0.12,
        n_iter_no_change=int(hypers.get("early_stopping_rounds", 50)),
    )
    hgb.fit(X_tr, y_tr)

joblib.dump(hgb, out / "best_model.pkl")

model_df["pred_final"] = hgb.predict(X_all)
oof_path = out / "oof_preds.csv"
model_df[["line_id", "env_id", "YEAR", "LOC", "YLDBE", "oof_pred", "pred_final"]].to_csv(
    oof_path, index=False
)

# Step 9 ranking
log_step("Step 9: Top-k per environment")
topk = int(cfg.get("topk", 10))
rank_rows = []
for env_id, chunk in model_df.loc[te_idx].groupby("env_id"):
    sub = chunk.sort_values("pred_final", ascending=False).head(topk).copy()
    sub["rank"] = np.arange(1, len(sub) + 1)
    sub["uncertainty_oof_rmse"] = glob_metrics["rmse"]
    rank_rows.append(sub)
rankings = pd.concat(rank_rows, axis=0) if rank_rows else pd.DataFrame()
rankings.to_csv(out / "rankings_topk_per_env.csv", index=False)

# Step 10 interpretation
log_step("Step 10: Permutation importance + buckets")
perm = permutation_importance(
    hgb,
    X_tr,
    y_tr,
    n_repeats=10,
    random_state=42,
    n_jobs=int(cfg.get("sklearn_n_jobs", 1)),
)
fi = pd.DataFrame({"feature": feature_cols, "importance_mean": perm.importances_mean, "importance_std": perm.importances_std})
fi = fi.sort_values("importance_mean", ascending=False)
fi.to_csv(out / "perm_importance.csv", index=False)


def bucket(name: str) -> str:
    if name.startswith("GPC_"):
        return "Genetic_PCA"
    if name.startswith("GXE__"):
        return "GxE_interaction"
    if any(x in name.upper() for x in ("PRCP", "TAVG", "HTDD", "CLDD", "DP01", "DP10")):
        return "Weather"
    if any(x in name.upper() for x in ("CLAY", "SILT", "SAND", "PH", "SOC", "NITROGEN", "CFVO")):
        return "Soil"
    if "past_yld" in name:
        return "Past_pheno"
    return "Other"


fi["bucket"] = fi["feature"].map(bucket)
bucketed = fi.groupby("bucket")["importance_mean"].sum().reset_index()
bucketed.to_csv(out / "perm_importance_buckets.csv", index=False)
fi.head(20).to_csv(out / "interpretation_top_features_slide.csv", index=False)

if cfg.get("use_shap", True):
    try:
        import shap

        explainer = shap.Explainer(hgb.predict, X_tr[: min(2000, len(X_tr))])
        sv = explainer(X_tr[: min(2000, len(X_tr))])
        shap.summary_plot(sv, feature_names=feature_cols, show=False)
        plt.tight_layout()
        plt.savefig(out / "shap.png", dpi=150, bbox_inches="tight")
        plt.close()
    except Exception as e:
        logging.warning("SHAP skipped: %s", e)

# Step 11 summary
summary = {
    "cohort": cohort,
    "join_report": join_report,
    "modeling_rows": int(len(model_df)),
    "oof_global_metrics": glob_metrics,
    "best_gbr_cv_params": search.best_params_,
    "feature_columns": feature_cols,
    "paths": {k: str(Path(v)) for k, v in paths.items()},
}


def _json_default(o: Any) -> Any:
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    return str(o)


with open(out / "run_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, default=_json_default)

logging.info("Done. Artifacts in %s", out)
print(json.dumps(summary, indent=2, default=_json_default))


## Saved Outputs / Artifacts

Artifacts are written under `output_dir` from your YAML config, including:

- `pipeline.log`
- `pheno_env.csv`
- `genotypes_<cohort>.csv`
- `full_dataset.csv` (if enabled)
- `env_scaler.pkl`
- `pca_pipeline.pkl`
- `ridge_snp_baseline.pkl`
- `env_gxe_feature_columns.csv`
- `metrics_by_env_oof.csv`
- `res_plot.png`
- `best_model.pkl`
- `oof_preds.csv`
- `rankings_topk_per_env.csv`
- `perm_importance.csv`
- `perm_importance_buckets.csv`
- `interpretation_top_features_slide.csv`
- `shap.png` (if SHAP enabled and available)
- `run_summary.json`

Next steps:
- Tune config values in `pipeline_config.yaml`
- Re-run phased cells from the desired stage
- Compare artifacts across cohort/config variants


## Yield Analysis by US Region

Group trials by agricultural regions using `LATITUDE`/`LONGITUDE` and `LOC` codes, then compare **actual vs predicted** average `YLDBE` (bu/ac).

This supports the hackathon narrative: **Corn Belt environments often show higher yields; regional residual patterns reveal GxE structure.**

In [16]:
# Cell 8 — Yield by US Region (geo EDA)
# Optional geo libs (fallbacks are built in)
try:
    import geopandas as gpd  # noqa: F401
except Exception:
    gpd = None

try:
    import plotly.express as px
except Exception:
    px = None

try:
    import seaborn as sns
except Exception:
    sns = None

from sklearn.cluster import KMeans

try:
    from IPython.display import display
except ImportError:
    display = print

# Training slice + OOF preds: pipeline stores OOF in model_df["oof_pred"], not a global oof_pred.
if "model_df" not in globals():
    raise NameError("Run Phase B (training) cells first so model_df exists.")

if "train_idx" in globals() and len(train_idx) == len(model_df):
    tr_df = model_df.loc[train_idx].copy()
elif "YEAR" in model_df.columns and "years" in globals() and isinstance(years, dict) and "train_min" in years and "train_max" in years:
    tr_df = model_df.loc[model_df["YEAR"].between(int(years["train_min"]), int(years["train_max"]))].copy()
else:
    tr_df = model_df.copy()

if "oof_preds" not in globals():
    oof_preds = None
    if "oof_pred" in globals() and isinstance(oof_pred, (np.ndarray, list, pd.Series)):
        oof_preds = np.asarray(oof_pred, dtype=float)
    elif "oof_train" in globals() and isinstance(oof_train, (np.ndarray, list)):
        oof_preds = np.asarray(oof_train, dtype=float)
    elif "oof_pred" in tr_df.columns:
        oof_preds = tr_df["oof_pred"].values.astype(float)
    elif "oof_pred" in model_df.columns:
        oof_preds = model_df.loc[tr_df.index, "oof_pred"].values.astype(float)
    elif "oof_full" in globals() and isinstance(oof_full, np.ndarray) and "train_idx" in globals():
        oof_preds = oof_full[np.where(train_idx)[0]]
    if oof_preds is None:
        raise ValueError(
            "No OOF predictions found. Run Phase B so model_df['oof_pred'] is populated (or define oof_train)."
        )
    if len(oof_preds) != len(tr_df):
        raise ValueError(
            f"OOF length {len(oof_preds)} does not match training rows {len(tr_df)}; re-run masks + Phase B."
        )

# Build a robust working frame from existing OOF context
if "full_df" in globals():
    region_df = full_df.copy()
else:
    region_df = pd.DataFrame(
        {
            "LOC": tr_df["LOC"].astype(str).values,
            "YEAR": tr_df["YEAR"].values,
            "YLDBE": tr_df["YLDBE"].values,
            "Pred": oof_preds,
        }
    )

if "YLDBE" not in region_df.columns:
    region_df["YLDBE"] = tr_df["YLDBE"].values
if "Pred" not in region_df.columns:
    region_df["Pred"] = oof_preds
if "LOC" not in region_df.columns:
    region_df["LOC"] = tr_df["LOC"].astype(str).values
if "YEAR" not in region_df.columns:
    region_df["YEAR"] = tr_df["YEAR"].values

# Ensure geocoordinates exist when available in model_df/tr_df
for c in ["LONGITUDE", "LATITUDE"]:
    if c not in region_df.columns:
        if c in tr_df.columns:
            region_df[c] = tr_df[c].values
        elif "model_df" in globals() and c in model_df.columns:
            # align by LOC+YEAR median location
            loc_year_geo = model_df.groupby(["LOC", "YEAR"], as_index=False)[c].median()
            region_df = region_df.merge(loc_year_geo, on=["LOC", "YEAR"], how="left")
        else:
            region_df[c] = np.nan

# 1) Region mapping by known site codes + fallback by state prefix
region_map = {
    "Corn Belt": ["IALI", "IAFO", "IASP", "IADA", "IAPR", "ILMN", "INVI"],
    "Northern Plains": ["NEDA", "MNHE", "MNOW", "SD"],
    "Central Plains": ["MOBU", "KS", "NE"],
    "South": ["TX", "OK", "AR", "LA", "MS", "AL", "GA", "FL"],
}


def infer_region(loc):
    s = str(loc).upper()
    for region, codes in region_map.items():
        if s in codes:
            return region
        if any(s.startswith(code) for code in codes):
            return region
    if s.startswith(("IA", "IL", "IN", "OH", "MO")):
        return "Corn Belt"
    if s.startswith(("MN", "ND", "SD")):
        return "Northern Plains"
    if s.startswith(("NE", "KS")):
        return "Central Plains"
    if s.startswith(("TX", "OK", "AR", "LA", "MS", "AL", "GA", "FL")):
        return "South"
    return "Other"


region_df["Region"] = region_df["LOC"].map(infer_region)

# Fallback: KMeans geo clusters when many sites remain uncategorized
other_frac = float((region_df["Region"] == "Other").mean())
if other_frac > 0.30 and region_df[["LONGITUDE", "LATITUDE"]].notna().all(axis=1).sum() >= 20:
    coords = region_df[["LONGITUDE", "LATITUDE"]].copy()
    coords = coords.fillna(coords.median(numeric_only=True)).fillna(0.0)
    km = KMeans(n_clusters=4, random_state=42, n_init=10)
    labels = km.fit_predict(coords.values)
    region_df["Region"] = [f"Region {chr(65 + i)}" for i in labels]

# 2) Summary tables
region_summary = (
    region_df.groupby(["Region", "YEAR"], as_index=False)
    .agg(
        Mean_Yield=("YLDBE", "mean"),
        Std_Yield=("YLDBE", "std"),
        N_Trials=("YLDBE", "count"),
        Mean_Pred=("Pred", "mean"),
    )
    .round(1)
)
print("Yield by Region/Year (bu/ac):")
display(region_summary)
region_summary.to_csv(out / "yield_by_region_year.csv", index=False)

region_overall = (
    region_df.groupby("Region", as_index=True)
    .agg(
        Mean_Yield=("YLDBE", "mean"),
        Std_Yield=("YLDBE", "std"),
        N_Trials=("YLDBE", "count"),
        Mean_Pred=("Pred", "mean"),
    )
    .round(1)
    .sort_values("Mean_Yield", ascending=False)
)
print("\nOverall by Region:")
display(region_overall)

# 3) Static visuals
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
region_overall[["Mean_Yield", "Mean_Pred"]].plot(kind="bar", ax=axes[0], width=0.6)
axes[0].set_title("Avg Yield: Actual vs Predicted by Region")
axes[0].set_ylabel("Yield (bu/ac)")
axes[0].legend(["Actual", "Pred"])

if sns is not None:
    sns.boxplot(data=region_df, x="Region", y="YLDBE", ax=axes[1])
else:
    region_df.boxplot(column="YLDBE", by="Region", ax=axes[1], grid=False)
    axes[1].set_title("Yield Distribution by Region")
    fig.suptitle("")
axes[1].set_title("Yield Distribution by Region")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(out / "yield_by_region.png", dpi=150, bbox_inches="tight")
plt.show()

# 4) Interactive geo map (if plotly available and coords present)
def _save_plotly_png(fig, path):
    try:
        fig.write_image(path, width=1200, height=800, scale=2)
        print("PNG map saved:", path)
    except Exception as e:
        print(f"PNG export skipped for {path.name}: {e} (install kaleido: pip install -U kaleido)")


if px is not None and region_df[["LATITUDE", "LONGITUDE"]].notna().all(axis=1).any():
    map_df = (
        region_df.dropna(subset=["LATITUDE", "LONGITUDE"])
        .groupby(["Region", "LOC"], as_index=False)
        .agg(
            LATITUDE=("LATITUDE", "mean"),
            LONGITUDE=("LONGITUDE", "mean"),
            Mean_Yield=("YLDBE", "mean"),
            Mean_Pred=("Pred", "mean"),
            N_Trials=("YLDBE", "count"),
        )
        .round(2)
    )
    size_col = "Mean_Yield"
    fig_map = px.scatter_geo(
        map_df,
        lat="LATITUDE",
        lon="LONGITUDE",
        color="Region",
        size=size_col,
        hover_data=["LOC", "Mean_Yield", "Mean_Pred", "N_Trials"],
        title="Trial Yields by US Region (Bubble = Avg Yield)",
        scope="usa",
    )
    fig_map.write_html(out / "yield_map.html")
    _save_plotly_png(fig_map, out / "yield_map.png")
    print("Map saved:", out / "yield_map.html")
else:
    print("Plotly map skipped (plotly missing or no valid LAT/LONG).")

print("Insight: Corn Belt should trend higher mean yields; regional spread indicates GxE complexity.")
print("Yield by US Region added – geo EDA for demo slides")


Yield by Region/Year (bu/ac):


,Region,YEAR,Mean_Yield,Std_Yield,N_Trials,Mean_Pred
0,Central Plains,2001,196.1,28.3,924,196.1
1,Central Plains,2002,194.2,35.8,429,194.2
2,Central Plains,2003,214.0,29.0,585,214.0
3,Central Plains,2004,222.8,27.0,1231,222.8
4,Central Plains,2005,214.4,33.1,964,214.4
5,Central Plains,2006,187.8,47.3,905,187.8
6,Central Plains,2007,205.7,31.7,620,205.7
7,Corn Belt,2001,181.4,36.1,2655,181.4
8,Corn Belt,2002,181.5,38.2,2633,181.4
9,Corn Belt,2003,178.0,35.4,3925,178.0



Overall by Region:


,Mean_Yield,Std_Yield,N_Trials,Mean_Pred
Region,,,,
Central Plains,206.5,35.9,5658,206.4
Corn Belt,192.7,34.9,24917,192.7
Northern Plains,189.3,34.2,6310,189.3
Other,183.6,39.7,6403,183.6
South,182.5,39.2,1712,182.5


C:\Users\kensm\AppData\Local\Temp\ipykernel_24172\2381979824.py:175: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


PNG export skipped for yield_map.png: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
 (install kaleido: pip install -U kaleido)
Map saved: C:\Users\kensm\Gods-Mercy\output\yield_map.html
Insight: Corn Belt should trend higher mean yields; regional spread indicates GxE complexity.
Yield by US Region added – geo EDA for demo slides
